<a href="https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My lane is **content decline / refresh prioritization**.

The goal of this baseline is to rank content items that deserve an editor's attention first.

I will build a simple human-readable rule before training any ML model. The rule will only use information available at the decision moment.

`trend_direction`, `trend_pct`, and the decline label will NOT be used to calculate the score. The label is used only afterward to evaluate whether the baseline ranked useful items.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


cwd = Path.cwd()

possible_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent
]

REPO_ROOT = cwd

for root in possible_roots:
    if (root / "data/raw/content_refresh_anonymized.csv").exists():
        REPO_ROOT = root
        break

repo_data = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"
uploaded_data = Path("content_refresh_anonymized.csv")

if repo_data.exists():
    DATA_PATH = repo_data
elif uploaded_data.exists():
    DATA_PATH = uploaded_data
else:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset:", DATA_PATH)
print("Shape:", df.shape)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("\nDecline label distribution:")
print(df["is_declining_label"].value_counts())

print("\nDecline base rate:")
print(round(df["is_declining_label"].mean(), 3))

Dataset: content_refresh_anonymized.csv
Shape: (14023, 44)

Decline label distribution:
is_declining_label
1    7542
0    6481
Name: count, dtype: int64

Decline base rate:
0.538


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The first signal I want to test is **days since the content was last updated**.

This signal is relevant to the refresh lane because older content may be more likely to need review or refreshing.

Before putting staleness into my rule, I want to check whether the measured decline rate actually changes across different staleness buckets.

I divide pages into:

- 0–30 days
- 31–90 days
- 91–180 days
- 181+ days

For each bucket I show `n`, because a rate without the number of observations behind it can be misleading.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, np.inf],
    labels=[
        "0-30",
        "31-90",
        "91-180",
        "181+"
    ]
)

staleness_table = (
    df.groupby(
        "staleness_bucket",
        observed=True
    )
    .agg(
        n=("content_id", "size"),
        decline_rate=("is_declining_label", "mean"),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

staleness_table["decline_rate"] = (
    staleness_table["decline_rate"] * 100
).round(2)

display(staleness_table)

,staleness_bucket,n,decline_rate,median_impressions
0,0-30,9603,51.10,475.0
1,31-90,78,60.26,567.5
2,91-180,4266,59.87,1750.0
3,181+,76,44.74,22.5


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

A ranked queue can look reasonable numerically while still contain bad recommendations.

I therefore inspect the top ten manually.

For every recommendation I ask three questions:

1. **Action:** what should the editor do?
2. **Why is it here?** which part of the rule caused the recommendation?
3. **What would make it wrong?** what real-world explanation could make this recommendation unnecessary?

The observed decline label is shown only to help evaluate the baseline. It was not used to create the ranking.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


NameError: name 'baseline_queue' is not defined

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

I do not assume that every highly ranked item is a good recommendation.

A useful baseline should expose its own weaknesses.

I inspect the top 20 for false positives — items prioritized by the rule but whose observed decline label is 0.

These cases help show where a simple hand-written rule can fail and what a later ML model would need to improve.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.